In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from scipy.ndimage import label
from tqdm import tqdm
import matplotlib.pyplot as plt
import os
import json
from collections import defaultdict
from matplotlib.backends.backend_pdf import PdfPages

MAX_STEPS = 10

def generate_percolation_lattice(size, p):
    """Generate a square lattice with site probability p"""
    return np.random.choice([0, 1], (size, size), p=[1-p, p]).astype(np.uint8)

def check_percolation(lattice):
    """Check for top-bottom and left-right percolation"""
    labeled, _ = label(lattice)
    top = set(labeled[0]) - {0}
    bottom = set(labeled[-1]) - {0}
    left = set(labeled[:,0]) - {0}
    right = set(labeled[:,-1]) - {0}
    top_bottom = float(bool(top & bottom))
    left_right = float(bool(left & right))
    return top_bottom, left_right

def first_coarse_graining(binary_lattice, dim):
    """Coarse grain lattice and duplicate for two channels"""
    t = torch.tensor(binary_lattice, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    patches = F.unfold(t, kernel_size=dim, stride=dim)
    patches = patches.permute(0, 2, 1)
    coarse_vals = patches.mean(dim=2)
    H, W = binary_lattice.shape
    new_h, new_w = H // dim, W // dim
    # Duplicate for two identical channels
    coarse_vals = coarse_vals.view(1, 1, new_h, new_w).repeat(1, 2, 1, 1)
    return coarse_vals.squeeze(0)

class DirectionalPercolationModel(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.rule = nn.Sequential(
            nn.Linear(2 * dim * dim, 64),  # 2*b^2 inputs
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 2),  # Two outputs
            nn.Sigmoid()
        )

    def forward(self, x, max_steps=MAX_STEPS):
        b, c, H, W = x.shape
        for _ in range(max_steps):
            if H < self.dim or W < self.dim:
                break
            patches = F.unfold(x, kernel_size=self.dim, stride=self.dim)
            patches = patches.permute(0, 2, 1).contiguous()
            # Process 2*b^2 inputs per patch
            out = self.rule(patches.view(-1, 2 * self.dim * self.dim))
            new_h, new_w = H // self.dim, W // self.dim
            # Reshape to two channels
            x = out.view(b, 2, new_h, new_w)
            _, _, H, W = x.shape
        
        # Final adaptive pooling to 1x1 to get two values
        x = F.adaptive_avg_pool2d(x, (1, 1))
        return x.view(b, 2)

def prepare_dataset(N, sizes):
    """Generate dataset with both uniform and critical samples"""
    data = []
    half = int(N/2)
    for _ in tqdm(range(half), desc="Generating uniform data"):
        p = np.random.uniform(0, 1)
        size = np.random.choice(sizes)
        L = generate_percolation_lattice(size, p)
        tb, lr = check_percolation(L)
        data.append((L, (tb, lr)))
    for _ in tqdm(range(half), desc="Generating critical data"):
        p = np.random.uniform(0.55, 0.65)
        size = np.random.choice(sizes)
        L = generate_percolation_lattice(size, p)
        tb, lr = check_percolation(L)
        data.append((L, (tb, lr)))
    return data

def train_epoch(model, device, data, batch_size, opt, crit, dim):
    """Train model for one epoch"""
    model.train()
    total_loss = 0.0
    for i in range(0, len(data), batch_size):
        batch = data[i:i+batch_size]
        groups = {}
        for x, y in batch:
            cg = first_coarse_graining(x, dim)
            size = tuple(cg.shape[-2:])
            groups.setdefault(size, []).append((cg, y))
        
        for size_key, group in groups.items():
            lattices, labels = zip(*group)
            inputs = torch.stack(lattices).to(device)
            # Convert labels to tensor [batch, 2]
            targets = torch.tensor(labels, dtype=torch.float32, device=device)
            opt.zero_grad()
            outputs = model(inputs)
            loss = crit(outputs, targets)
            loss.backward()
            opt.step()
            total_loss += loss.item() * len(group)
    
    return total_loss / len(data)

def test_systems(model, dim, power, device='cpu',
                 num_tests=50, system_size='standard',
                 p_range=(0,1), verbose=True):
    """Test model on various system sizes"""
    model.to(device).eval()
    mapping = {
        f'{dim}^2': power - 1,
        f'{dim}^3': power,
        f'{dim}^4': power + 1,
        f'{dim}^5': power + 2,
        f'{dim}^6': power + 3,
        f'{dim}^7': power + 4
    }
    size_power = mapping[system_size]
    L = dim ** size_power

    results = []
    for _ in range(num_tests):
        p = np.random.uniform(*p_range)
        raw = generate_percolation_lattice(L, p)
        tb, lr = check_percolation(raw)
        lbl = 1.0 if (tb or lr) else 0.0  # Overall percolation
        coarse = first_coarse_graining(raw, dim)
        inp = coarse.unsqueeze(0).to(device)
        
        with torch.no_grad():
            out = model(inp).squeeze(0)
            out1, out2 = out[0].item(), out[1].item()
            overall_pred = 1 - (1 - out1) * (1 - out2)  # OR of both channels
        
        results.append((raw, lbl, overall_pred))

    acc = sum((pred > 0.5) == lbl for _, lbl, pred in results) / num_tests
    pos = [pred for _, lbl, pred in results if lbl == 1]
    neg = [pred for _, lbl, pred in results if lbl == 0]
    
    metrics = {
        'accuracy': acc,
        'avg_pred_perc': np.mean(pos) if pos else 0,
        'avg_pred_non_perc': np.mean(neg) if neg else 0
    }
    
    if verbose:
        print(f"\nAfter manual first coarse-grain -> NN cascade on {L}×{L}:")
        print(f" Accuracy        : {acc:.2%}")
        print(f" Avg pred | Perc     : {metrics['avg_pred_perc']:.3f}")
        print(f" Avg pred | Non-Perc : {metrics['avg_pred_non_perc']:.3f}")

    return metrics

def visualize_rule(model, dim, device='cpu', num_samples=100):
    """Visualize rule with overall percolation probability"""
    p_values = np.linspace(0, 1, 100)
    model.eval()
    overall_outputs = []
    with torch.no_grad():
        for p in p_values:
            # Create input for the rule: [num_samples, 2 * dim * dim]
            input_vec = torch.full((num_samples, 2 * dim * dim), p, 
                                  dtype=torch.float32, device=device)
            # Get deterministic output
            outputs = model.rule(input_vec)
            # Compute overall percolation probability
            overall = 1 - (1 - outputs[:,0]) * (1 - outputs[:,1])
            overall_outputs.append(overall.mean().item())
    
    overall_outputs = np.array(overall_outputs)
    diff = overall_outputs - p_values
    crossings = []
    for i in range(len(p_values) - 1):
        if p_values[i] < 0.1 or p_values[i] > 0.9:
            continue
        if diff[i] * diff[i+1] <= 0:
            x1, x2 = p_values[i], p_values[i+1]
            y1, y2 = diff[i], diff[i+1]
            if y1 == y2:
                continue
            cross = x1 - y1 * (x2 - x1) / (y2 - y1)
            crossings.append(cross)
    
    valid_mask = (p_values >= 0.1) & (p_values <= 0.9)
    valid_p = p_values[valid_mask]
    valid_out = overall_outputs[valid_mask]
    abs_diff = np.abs(valid_out - valid_p)
    if len(valid_p) > 0:
        p_c_model = valid_p[np.argmin(abs_diff)]
    else:
        p_c_model = p_values[np.argmin(np.abs(overall_outputs - p_values))]
    
    idx = np.argmin(np.abs(p_values - p_c_model))
    
    if idx == 0:
        deriv = (overall_outputs[1] - overall_outputs[0]) / (p_values[1] - p_values[0])
    elif idx == len(p_values) - 1:
        deriv = (overall_outputs[-1] - overall_outputs[-2]) / (p_values[-1] - p_values[-2])
    else:
        deriv = (overall_outputs[idx+1] - overall_outputs[idx-1]) / (p_values[idx+1] - p_values[idx-1])
    
    lambda_val = abs(deriv)
    if lambda_val > 1:
        nu = np.log(dim) / np.log(lambda_val)
    else:
        nu = float('nan')
    
    return p_values, overall_outputs, p_c_model, crossings, deriv, nu

def run_experiment(dim, power, num_runs, device, ratio, epochs):
    """Run full experiment for given configuration"""
    all_rule_curves = []
    all_test_results = defaultdict(list)
    all_pc_values = []
    all_nu_values = []
    
    # Calculate model parameters to determine training samples
    model = DirectionalPercolationModel(dim).to(device)
    num_params = sum(p.numel() for p in model.parameters())
    train_samples = int(num_params / ratio)
    
    print(f"\nStarting experiment for DIM={dim}")
    print(f"Parameters: {num_params}, Ratio: {ratio}, Training samples: {train_samples}")
    
    for run_idx in range(num_runs):
        print(f"\n{'='*40}")
        print(f"Run {run_idx+1}/{num_runs} for DIM={dim}, Ratio={ratio}")
        print(f"{'='*40}")
        
        sizes = [dim**2, dim**3, dim**4]
        train_data = prepare_dataset(train_samples, sizes)
        
        model = DirectionalPercolationModel(dim).to(device)
        opt = optim.Adam(model.parameters(), lr=1e-3)
        crit = nn.BCELoss()
        
        # Training loop
        for epoch in range(1, epochs + 1):
            loss = train_epoch(model, device, train_data, 10, opt, crit, dim)
            if epoch % 50 == 0 or epoch == 1 or epoch == epochs:
                print(f"Epoch {epoch}/{epochs} — Loss: {loss:.4f}")
        
        # Test configurations
        test_configs = [
            {'system_size': f'{dim}^2', 'num_tests': 100, 'p_range': (0.55, 0.65)},
            {'system_size': f'{dim}^3', 'num_tests': 100, 'p_range': (0.55, 0.65)},
            {'system_size': f'{dim}^4', 'num_tests': 100, 'p_range': (0.55, 0.65)},
            {'system_size': f'{dim}^5', 'num_tests': 100, 'p_range': (0.55, 0.65)},
        ]
        
        for config in test_configs:
            key = f"{config['system_size']}_{config['p_range'][0]}-{config['p_range'][1]}"
            metrics = test_systems(
                model, dim, power, device,
                num_tests=config['num_tests'],
                system_size=config['system_size'],
                p_range=config['p_range'],
                verbose=False
            )
            all_test_results[key].append(metrics)
        
        p_vals, outputs, p_c_model, crossings, deriv, nu = visualize_rule(model, dim, device)
        all_rule_curves.append((p_vals, outputs, p_c_model, crossings))
        all_pc_values.append(p_c_model)
        all_nu_values.append(nu)
        
        print(f"Critical point estimate: {p_c_model:.4f}")
        print(f"Critical exponent ν: {nu:.4f}")
        
        del model, opt, crit, train_data
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    
    return all_rule_curves, all_test_results, all_pc_values, all_nu_values

def save_consolidated_results_for_dim_ratio(dim, ratio, results):
    """Save results for specific dimension and ratio"""
    all_rule_curves, all_test_results, all_pc_values, all_nu_values = results
    
    # Create directory structure
    base_dir = f"results_dim_{dim}"
    ratio_dir = os.path.join(base_dir, f"ratio_{ratio}")
    os.makedirs(ratio_dir, exist_ok=True)
    
    # Save PDF report
    pdf_path = os.path.join(ratio_dir, "consolidated_results.pdf")
    with PdfPages(pdf_path) as pdf:
        valid_nu = [nu for nu in all_nu_values if not np.isnan(nu)]
        avg_pc = np.mean(all_pc_values)
        std_pc = np.std(all_pc_values)
        avg_nu = np.mean(valid_nu) if valid_nu else float('nan')
        std_nu = np.std(valid_nu) if valid_nu else float('nan')
        
        plt.figure(figsize=(10, 6))
        for i, (p_vals, outputs, p_c_model, crossings) in enumerate(all_rule_curves):
            plt.plot(p_vals, outputs, alpha=0.5, color='blue')
            valid_crossings = [cross for cross in crossings if 0.1 <= cross <= 0.9]
            for cross in valid_crossings:
                plt.scatter(cross, cross, color='black', s=30, zorder=3)
        
        plt.plot([0, 1], [0, 1], 'k--', label=r'$f(p) = p$', linewidth=1.5)
        plt.plot([], [], ' ', label=f'$p_c = {avg_pc:.3f} \\pm {std_pc:.3f}$')
        if not np.isnan(avg_nu):
            plt.plot([], [], ' ', label=f'$\\nu = {avg_nu:.3f} \\pm {std_nu:.3f}$')
        
        plt.xlabel('$p$', fontsize=16)
        plt.ylabel(r'$\mathcal{Q}(p)$', fontsize=16)
        plt.title(f"Rule Projection - DIM={dim}, Ratio={ratio}", fontsize=20)
        plt.legend(fontsize=13)
        plt.xlim(0, 1)
        plt.ylim(0, 1)
        pdf.savefig(bbox_inches='tight')
        plt.close()
    
    # Save text report
    txt_path = os.path.join(ratio_dir, "consolidated_results.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(f"{'='*40}\n")
        f.write(f"Critical Point Analysis (DIM={dim}, Ratio={ratio})\n")
        f.write(f"{'='*40}\n\n")
        f.write("p_c values from each run:\n")
        for i, pc in enumerate(all_pc_values):
            f.write(f"Run {i+1}: {pc:.6f}\n")
        
        f.write(f"\nAverage p_c: {np.mean(all_pc_values):.6f}\n")
        f.write(f"Standard deviation: {np.std(all_pc_values):.6f}\n\n")
        
        f.write(f"Critical Exponent ν values:\n")
        for i, nu in enumerate(all_nu_values):
            status = "VALID" if not np.isnan(nu) else "INVALID (λ≤1)"
            f.write(f"Run {i+1}: {nu:.6f} [{status}]\n")
        
        if valid_nu:
            f.write(f"\nAverage ν (valid runs): {np.mean(valid_nu):.6f}\n")
            f.write(f"Standard deviation: {np.std(valid_nu):.6f}\n")
        else:
            f.write("\nNo valid ν values (all runs had λ≤1)\n")
        f.write("\n")
        
        f.write(f"{'='*40}\n")
        f.write(f"Test Performance Metrics (DIM={dim}, Ratio={ratio})\n")
        f.write(f"{'='*40}\n\n")
        
        for config, results_list in all_test_results.items():
            parts = config.split('_')
            system_size = parts[0]
            p_range = parts[1]
            
            accuracies = [r['accuracy'] for r in results_list]
            avg_perc = [r['avg_pred_perc'] for r in results_list]
            avg_non_perc = [r['avg_pred_non_perc'] for r in results_list]
            
            f.write(f"System: {system_size}, p-range: {p_range}\n")
            f.write("-"*50 + "\n")
            
            f.write("Run | Accuracy | Avg Perc | Avg Non-Perc\n")
            f.write("----|----------|----------|------------\n")
            for i in range(len(results_list)):
                f.write(f"{i+1:3d} | {accuracies[i]:.4f} | {avg_perc[i]:.4f} | {avg_non_perc[i]:.4f}\n")
            
            f.write("\nSummary Statistics:\n")
            f.write(f"Accuracy: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}\n")
            f.write(f"Avg Perc: {np.mean(avg_perc):.4f} ± {np.std(avg_perc):.4f}\n")
            f.write(f"Avg Non-Perc: {np.mean(avg_non_perc):.4f} ± {np.std(avg_non_perc):.4f}\n")
            f.write("="*50 + "\n\n")
        f.write("\n\n")

def main():
    """Main function to run all experiments"""
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    POWER = 3
    
    # Define experiment parameters: {dim: {ratio: epochs}}
    EXPERIMENT_PARAMS = {
        3: {0.1: 200,
            0.2: 100,
            0.5: 200
        },
        4: {0.1: 800,
            0.2: 800,
            0.5: 800
        },
        5: {0.1: 800,
            0.2: 500,
            0.5: 600
        }
    }
    
    # Run experiments for each dimension and ratio
    for dim, ratio_epochs in EXPERIMENT_PARAMS.items():
        print(f"\n\n{'='*50}")
        print(f"STARTING EXPERIMENTS FOR DIM={dim}")
        print(f"{'='*50}")
        
        for ratio, epochs in ratio_epochs.items():
            print(f"\n{'='*50}")
            print(f"STARTING RUNS FOR DIM={dim}, RATIO={ratio}")
            print(f"{'='*50}")
            
            results = run_experiment(
                dim=dim,
                power=POWER,
                num_runs=1,
                device=DEVICE,
                ratio=ratio,
                epochs=epochs
            )
            
            save_consolidated_results_for_dim_ratio(dim, ratio, results)

if __name__ == "__main__":
    main()



STARTING EXPERIMENTS FOR DIM=3

STARTING RUNS FOR DIM=3, RATIO=0.1

Starting experiment for DIM=3
Parameters: 3858, Ratio: 0.1, Training samples: 38580

Run 1/1 for DIM=3, Ratio=0.1


Generating critical data: 100%|█████████████████████████████████████████████████████████████| 19290/19290 [00:03<00:00, 5573.80it/s]


Epoch 1/200 — Loss: 0.3722
Epoch 50/200 — Loss: 0.2870
Epoch 100/200 — Loss: 0.2838
Epoch 150/200 — Loss: 0.2810
Epoch 200/200 — Loss: 0.2803
Critical point estimate: 0.4747
Critical exponent ν: 0.8873

STARTING RUNS FOR DIM=3, RATIO=0.2

Starting experiment for DIM=3
Parameters: 3858, Ratio: 0.2, Training samples: 19290

Run 1/1 for DIM=3, Ratio=0.2


Generating critical data: 100%|███████████████████████████████████████████████████████████████| 9645/9645 [00:01<00:00, 5507.73it/s]


Epoch 1/100 — Loss: 0.4113
Epoch 50/100 — Loss: 0.2930
Epoch 100/100 — Loss: 0.2857
Critical point estimate: 0.3838
Critical exponent ν: 1.6801

STARTING RUNS FOR DIM=3, RATIO=0.5

Starting experiment for DIM=3
Parameters: 3858, Ratio: 0.5, Training samples: 7716

Run 1/1 for DIM=3, Ratio=0.5


Generating critical data: 100%|███████████████████████████████████████████████████████████████| 3858/3858 [00:00<00:00, 5518.24it/s]


Epoch 1/200 — Loss: 0.4681
Epoch 50/200 — Loss: 0.2898
Epoch 100/200 — Loss: 0.2774
Epoch 150/200 — Loss: 0.2728
Epoch 200/200 — Loss: 0.2628
Critical point estimate: 0.5556
Critical exponent ν: 0.5289


STARTING EXPERIMENTS FOR DIM=4

STARTING RUNS FOR DIM=4, RATIO=0.1

Starting experiment for DIM=4
Parameters: 4754, Ratio: 0.1, Training samples: 47540

Run 1/1 for DIM=4, Ratio=0.1


Generating critical data: 100%|█████████████████████████████████████████████████████████████| 23770/23770 [00:19<00:00, 1208.48it/s]


Epoch 1/800 — Loss: 0.3427
